# **Inferencia binaria: HybridCNN Audio Classifier (`alertable`)**

Carga el modelo entrenado y predice sobre audios `.wav` / `.mp3` usando **exactamente el mismo pipeline** que `preprocess.py → main()` a través de `Preprocess.process_audio_file()`.

Soporta los modos `mel_only`, `mel_mfcc` y `mel_waveform` con detección automática desde el state_dict.

> **`AUGMENT_INFERENCE`:** activa aumentaciones de dominio (ruido, EQ telefónico, reverb sintética) sobre audios limpios externos para acercarlos al dominio de entrenamiento. Hay que establecerlo como `True` cuando usemos audios descargados de Internet.

## **1. Importaciones**

In [23]:
from __future__ import annotations

import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import soundfile as sf
import torchaudio.transforms as T

from tqdm import tqdm # en vez de from tqdm.notebook import tqdm
from pathlib import Path
from src.utils.config import *
from src.models.hybrid_cnn_v3 import ImprovedMFCCCNN
from src.models.hybrid_cnn_v4 import HybridCNNV5
from src.data.preprocess import Preprocess, PreprocessConfig # pipeline centralizado

print("Importaciones completadas.")

Importaciones completadas.


## **2. Configuración**

In [24]:
# ─────────────────────────────────────────────
# RUTAS
# ─────────────────────────────────────────────
MODEL_PATH = FINAL_MODEL_DIR / "best_alertable_v4.pt"
# MODEL_PATH = CHECKPOINT_DIR / "alertable_V4" / "checkpoint_epoch_15.pt"

PROCESSED_METADATA = PROCESSED_METADATA['2']
LABEL_MAPPING_PATH = LABEL_MAPPING["alertable2"]

AUDIO_FOLDER = TEST_AUDIO_FOLDER # carpeta con .wav / .mp3

# ─────────────────────────────────────────────
# PARÁMETROS DE AUDIO  (igual que preprocess.py → main())
# ─────────────────────────────────────────────
SAMPLE_RATE = 16000
N_MELS = 128
N_MFCC = 13
N_FFT = 1024
HOP_LENGTH = 160
PEAK_TARGET = 0.99

# ─────────────────────────────────────────────
# MODELO
# ─────────────────────────────────────────────
DROPOUT = 0.35

# ─────────────────────────────────────────────
# INFERENCIA
# ─────────────────────────────────────────────
TOP_K = 2 # binario: solo 2 clases

# ─────────────────────────────────────────────
# AUGMENTACIÓN DE DOMINIO EN INFERENCIA
# ─────────────────────────────────────────────
# True cuando el audio venga de Internet / estudio (limpio).
# False si el audio ya proviene del mismo dominio del dataset.
AUGMENT_INFERENCE = True

print(f"- Carpeta de audios: {AUDIO_FOLDER}")
print(f"- Modelo: {MODEL_PATH}")
print(f"- Label mapping: {LABEL_MAPPING_PATH}")

- Carpeta de audios: /home/andres/Documentos/proyecto4geeks/tests/audios
- Modelo: /home/andres/Documentos/proyecto4geeks/models/final/best_alertable_v4.pt
- Label mapping: /home/andres/Documentos/proyecto4geeks/data/interim/processed_dataset/label_mapping_alertableV2.pkl


## **3. Cargar label mapping**

In [25]:
def load_label_mapping(path: Path) -> tuple[dict, dict, int]:
    """Devuelve (label2idx, idx2label, num_classes)."""
    with open(path, "rb") as f:
        payload = pickle.load(f)

    if isinstance(payload, dict) and "label2idx" in payload:
        label2idx = payload["label2idx"]
        idx2label = payload["idx2label"]
    else:
        label2idx = payload
        idx2label = {v: k for k, v in label2idx.items()}

    num_classes = len(label2idx)
    return label2idx, idx2label, num_classes

label2idx, idx2label, NUM_CLASSES = load_label_mapping(LABEL_MAPPING_PATH)

print(f"{NUM_CLASSES} clases cargadas")
print(f"Clases: {list(label2idx.keys())}")

2 clases cargadas
Clases: [False, True]


## **4. Instanciar `Preprocess` (pipeline centralizado)**

- En lugar de construir los transforms manualmente, se delega en `Preprocess`.
- Los parámetros deben ser **idénticos** a los usados en `preprocess.py → main()`.

In [26]:
# Instanciar Preprocess con los mismos parámetros que preprocess.py → main()
# target_duration=None → sin recorte (comportamiento de inferencia)
pp_config = PreprocessConfig(
    sample_rate=SAMPLE_RATE,
    target_duration=None, # sin padding/recorte en inferencia
    normalize_peak=True,
    peak_target=PEAK_TARGET,
    n_mels=N_MELS,
    n_mfcc=N_MFCC,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    save_audio=False,
    save_mel=False,
    save_mfcc=False,
    save_waveform=False,
    augment_inference=AUGMENT_INFERENCE,
)

pp = Preprocess(config=pp_config)
device = pp.device

print(f"Dispositivo: {device}")
print(f"AUGMENT_INFERENCE: {AUGMENT_INFERENCE}")
print("Preprocess instanciado - transforms listos")

Dispositivo: cuda
AUGMENT_INFERENCE: True
Preprocess instanciado - transforms listos


## **5. Pipeline de preprocesado - delegado en `Preprocess.process_audio_file()`**

`Preprocess.process_audio_file(path)` aplica exactamente el mismo pipeline que `preprocess.py → main()`: 

mono → resampleo → fix_length → normalización de pico → mel + mfcc.

In [27]:
def load_and_preprocess(
    audio_path: Path,
    augment_inference: bool = AUGMENT_INFERENCE,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Wrapper que delega en Preprocess.process_audio_file().
    Devuelve (mel, mfcc, waveform) con shape:
      mel: [1, N_MELS, T] float32
      mfcc: [1, N_MFCC, T] float32
      waveform: [1, T] float32 - necesario para el modo mel_waveform
    """
    mel, mfcc = pp.process_audio_file(audio_path, augment_inference=augment_inference)
    # Obtenemos waveform por separado para mel_waveform
    # process_audio_file ya aplica todo el pipeline; recalculamos solo el waveform

    audio_np, sr = sf.read(str(audio_path))
    waveform = torch.tensor(audio_np, dtype=torch.float32)
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    else:
        waveform = waveform.transpose(0, 1)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.to(device)
    if sr != SAMPLE_RATE:
        resampler = T.Resample(orig_freq=sr, new_freq=SAMPLE_RATE).to(device)
        waveform = resampler(waveform)
    peak = waveform.abs().max().clamp_min(1e-8)
    waveform = (waveform / peak * PEAK_TARGET).float()

    return mel.float(), mfcc.float(), waveform

print("load_and_preprocess() listo (delega en Preprocess)")

load_and_preprocess() listo (delega en Preprocess)


## **6. Cargar el modelo**

In [28]:
def detect_mode_from_state_dict(state_dict: dict) -> str:
    keys = set(state_dict.keys())
    has_mel = any(k.startswith("cnn_mel.") for k in keys)
    has_mfcc = any(k.startswith("cnn_mfcc.") for k in keys)
    has_waveform = any(k.startswith("cnn_wave.") for k in keys)

    if has_mel and has_waveform:
        return "mel_waveform"
    elif has_mel and has_mfcc:
        return "mel_mfcc"
    elif has_mel:
        return "mel_only"
    elif has_mfcc:
        return "mfcc_only"
    else:
        raise ValueError("No se encontraron keys cnn_mel.*, cnn_mfcc.* ni waveform_cnn.* en el state_dict.")

def load_model(model_path: Path, num_classes: int, dropout: float) -> tuple[HybridCNNV5, str]:
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)

    if isinstance(checkpoint, dict) and "model_state" in checkpoint:
        state_dict = checkpoint["model_state"]
        epoch_info = checkpoint.get("epoch", "?")
        acc_info = checkpoint.get("best_acc", "?")
        print(f"Checkpoint - epoch: {epoch_info} | best_acc: {acc_info}")
    else:
        state_dict = checkpoint

    detected_mode = detect_mode_from_state_dict(state_dict)
    print(f"Modo detectado automáticamente: {detected_mode}")

    model = HybridCNNV5(num_classes=num_classes, dropout=dropout, mode=detected_mode).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, detected_mode

model, MODE = load_model(MODEL_PATH, NUM_CLASSES, DROPOUT)

total_params = sum(p.numel() for p in model.parameters())
print(f"Modelo cargado | {total_params:,} parámetros | modo: {MODE}")

Modo detectado automáticamente: mel_waveform
Modelo cargado | 1,807,105 parámetros | modo: mel_waveform


## **7. Función de predicción**

In [29]:
@torch.inference_mode()
def predict(audio_path: Path, top_k: int = 2) -> dict:

    mel, mfcc, waveform = load_and_preprocess(audio_path)

    mel_b = mel.unsqueeze(0).to(device)

    mfcc_b = None
    waveform_b = None

    if mfcc is not None:
        mfcc_b = mfcc.unsqueeze(0).to(device)

    if waveform is not None:
        waveform_b = waveform.unsqueeze(0).to(device)

    # Forward
    if MODE == "mel_only":
        logits = model(mel=mel_b)

    elif MODE == "mfcc_only":
        logits = model(mel=mel_b, mfcc=mfcc_b)

    elif MODE == "mel_mfcc":
        logits = model(mel=mel_b, mfcc=mfcc_b)

    elif MODE == "mel_waveform":
        logits = model(mel=mel_b, waveform=waveform_b)

    else:
        raise ValueError(f"Modo desconocido: {MODE}")

    # =========================================================
    # BINARIO
    # =========================================================

    prob_alertable = torch.sigmoid(logits).item()
    prob_no_alertable = 1.0 - prob_alertable

    probs_dict = {
        "alertable": prob_alertable,
        "no_alertable": prob_no_alertable,
    }

    # Ordenar por probabilidad descendente
    top_list = sorted(
        probs_dict.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    pred_label = top_list[0][0]
    pred_conf = float(top_list[0][1])

    return {
        "filename": audio_path.name,

        "prediction": pred_label,

        "confidence": pred_conf,

        "top_k": [
            (label, float(prob))
            for label, prob in top_list
        ],

        "probabilities": {
            "alertable": float(prob_alertable),
            "no_alertable": float(prob_no_alertable),
        },

        "logit_raw": float(logits.item()),
    }

## **8. Probar un audio individual**

In [30]:
# ────────────────────────────────────────────────────────────────────────
# SINGLE_AUDIO = Path("/ruta/al/audio.wav")
# ────────────────────────────────────────────────────────────────────────

# Demo: el primer audio de la carpeta, si existe
audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if audios:
    SINGLE_AUDIO = audios[0]
    result = predict(SINGLE_AUDIO)

    print(f"\nArchivo: {result['filename']}")
    print(f"Predicción: {result['prediction']}")
    print(f"Confianza: {result['confidence']:.2%}")
    print(f"\nTop-{TOP_K}:")
else:
    print(f"No hay audios .wav/.mp3 en {AUDIO_FOLDER}")


Archivo: 11325622-police-siren-sound-effect-240674.mp3
Predicción: alertable
Confianza: 99.69%

Top-2:


## **9. Inferencia por lotes sobre toda la carpeta**

In [31]:
audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if not audios:
    raise FileNotFoundError(f"No se encontraron audios en {AUDIO_FOLDER}")

print(f"{len(audios)} audios encontrados en {AUDIO_FOLDER}\n")

rows = []
errors = []
prediction = {"ok": 0, "no_ok": 0}

for audio_path in tqdm(audios, desc="Procesando audios"):
    try:
        result = predict(audio_path)

        # Validación por estructura de carpetas:
        # .../alertables/audio.wav → esperamos True
        # .../no_alertables/audio.wav → esperamos False
        folder_name = audio_path.parent.name.lower()
        if folder_name in ("alertables", "alertable"):
            expected = "alertable"
        elif folder_name in ("no_alertables", "no_alertable"):
            expected = "no_alertable"
        else:
            expected = None # carpeta sin etiqueta conocida

        if expected is not None:
            if result["prediction"] == expected:
                prediction["ok"] += 1
            else:
                prediction["no_ok"] += 1

        row = {
            "audio_path": str(audio_path.parent.name),
            "filename": result["filename"],
            "prediction": result["prediction"],
            "confidence": result["confidence"],
            "expected": expected,
            "correct": (result["prediction"] == expected) if expected is not None else None,
        }
        for label, prob in result["top_k"]:
            row[f"prob_{label}"] = round(prob, 4)
        rows.append(row)

    except Exception as e:
        errors.append({"filename": audio_path.name, "error": str(e)})
        print(f"Error en {audio_path.name}: {e}")

results_df = pd.DataFrame(rows)

total_labeled = prediction["ok"] + prediction["no_ok"]
acc = prediction["ok"] / total_labeled if total_labeled > 0 else float("nan")

print(f"\nProcesados: {len(rows)} | Errores: {len(errors)}")
print(f"Correctos: {prediction['ok']} / {total_labeled} | ({acc:.2%})")
print(f"Incorrectos: {prediction['no_ok']}")
results_df.head(40)

35 audios encontrados en /home/andres/Documentos/proyecto4geeks/tests/audios



Procesando audios: 100%|██████████| 35/35 [00:01<00:00, 25.02it/s]


Procesados: 35 | Errores: 0
Correctos: 28 / 35 | (80.00%)
Incorrectos: 7


,audio_path,filename,prediction,confidence,expected,correct,prob_alertable,prob_no_alertable
0,alertables,11325622-police-siren-sound-effect-240674.mp3,alertable,0.999805,alertable,True,0.9998,0.0002
1,alertables,ElevenLabs_A_6_to_7-year-old_child_crying_and_...,no_alertable,0.657166,alertable,False,0.3428,0.6572
2,alertables,audio_607a0.mp3,no_alertable,0.741110,alertable,False,0.2589,0.7411
3,alertables,child-crime-aw2xrhhk.wav,alertable,0.978249,alertable,True,0.9782,0.0218
4,alertables,dragon-studio-car-crash-sound-effect-376874.mp3,alertable,0.544209,alertable,True,0.5442,0.4558
5,alertables,dragon-studio-dog-barking-406629.mp3,alertable,0.999960,alertable,True,1.0000,0.0000
6,alertables,explosion-meme_dTCfAHs.mp3,alertable,0.990201,alertable,True,0.9902,0.0098
7,alertables,freesound_community-car-crash-edit-two-92001.mp3,no_alertable,0.820695,alertable,False,0.1793,0.8207
8,alertables,freesound_community-dog-barking-70772.mp3,alertable,0.993048,alertable,True,0.9930,0.0070
9,alertables,freesound_community-glass-shatter-7-95202.mp3,no_alertable,0.649799,alertable,False,0.3502,0.6498


## **10. Resumen de predicciones**

In [32]:
if not results_df.empty:
    summary = (
        results_df
        .groupby("prediction")
        .agg(
            count=("filename", "count"),
            avg_confidence=("confidence", "mean"),
            correct=("correct", lambda x: x.sum() if x.notna().any() else None),
        )
        .sort_values("count", ascending=False)
        .reset_index()
    )
    summary["avg_confidence"] = summary["avg_confidence"].map("{:.2%}".format)
    print("Distribución de predicciones:")
    display(summary)

    # Archivos con confianza baja
    CONFIDENCE_THRESHOLD = 0.60
    low_conf = results_df[results_df["confidence"] < CONFIDENCE_THRESHOLD]
    if not low_conf.empty:
        print(f"\n{len(low_conf)} audios con confianza < {CONFIDENCE_THRESHOLD:.0%}:")
        display(low_conf[["filename", "prediction", "confidence", "expected"]])

    # Falsos negativos (alertable predicho como no alertable)
    if "expected" in results_df.columns:
        fn = results_df[(results_df["expected"] == True) & (results_df["prediction"] == False)]
        fp = results_df[(results_df["expected"] == False) & (results_df["prediction"] == True)]
        if not fn.empty:
            print(f"\nFalsos negativos (alertable → no alertable): {len(fn)}")
            display(fn[["filename", "confidence"]].head(10))
        if not fp.empty:
            print(f"\nFalsos positivos (no alertable → alertable): {len(fp)}")
            display(fp[["filename", "confidence"]].head(10))

Distribución de predicciones:


,prediction,count,avg_confidence,correct
0,no_alertable,24,87.27%,17
1,alertable,11,87.90%,11



2 audios con confianza < 60%:


,filename,prediction,confidence,expected
4,dragon-studio-car-crash-sound-effect-376874.mp3,alertable,0.544209,alertable
30,freesound_community-print-shop-printer-3-23680...,no_alertable,0.559904,no_alertable


## **11. Exportar resultados a CSV**

In [33]:
OUTPUT_CSV = AUDIO_FOLDER / "predictions.csv"

if not results_df.empty:
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Resultados guardados en: {OUTPUT_CSV}")
else:
    print("No hay resultados para exportar.")

Resultados guardados en: /home/andres/Documentos/proyecto4geeks/tests/audios/predictions.csv
